In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
import os
import random
from tqdm import tqdm
import multiprocessing
from rdkit import Chem

sys.path.append("/home/iscb/wolfson/hagairavid/LocAlign")

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"ligand_id",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]


In [ ]:
# Reproducibility: fix the RNG seed used by the random sampling steps below
# (ligand-combination subsampling, probability-weighted final sampling), and
# make every candidate-filtering step persist what it drops to
# calculated_cath/exclusion_logs/ instead of only printing row counts.
# NOTE: earlier runs of this notebook that produced the currently-released
# manifests did not set this seed, so this makes *future* runs of this
# notebook reproducible; it does not retroactively reproduce the exact
# candidate sample behind the already-released CSVs (those are released
# verbatim as datasets/csv_files/*).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXCLUSION_LOG_DIR = os.path.join('calculated_cath', 'exclusion_logs')
os.makedirs(EXCLUSION_LOG_DIR, exist_ok=True)


def log_excluded(excluded_df, reason: str, filename: str) -> None:
    if len(excluded_df) == 0:
        return
    out = excluded_df.copy()
    out['exclusion_reason'] = reason
    out.to_csv(os.path.join(EXCLUSION_LOG_DIR, filename), index=False)
    print(f"Logged {len(out)} excluded rows ({reason}) to {os.path.join(EXCLUSION_LOG_DIR, filename)}")


In [ ]:
# df = pd.read_csv('/home/iscb/wolfson/hagairavid/databases/BioLiP_nr_0102.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)
df = pd.read_csv('/home/iscb/wolfson/hagairavid/LocAlign/BioLiP_nr_10082025.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)
# change Ligand_ID to ligand_id
df = df.rename(columns={"Ligand_ID": "ligand_id"})

relevant_columns = ["PDB ID", "Receptor chain", "ligand_id", "Ligand_chain"]
df = df[relevant_columns].astype('str')

### Ligand metdata file
Contains information regard each ligand, including the SMILE structure that will be later use to count the number of atoms per ligand

In [ ]:
tsv_file = 'ligand.tsv.gz'
# tsv_file = '/home/iscb/wolfson/hagairavid/LocAlign/ligand_nr.tar.bz2'
ligands_df = pd.read_csv(tsv_file, sep='\t', encoding='utf-8', on_bad_lines='skip')
ligands_df.set_index(ligands_df.columns[0], inplace=True)
ligands_df.to_csv('ligands.csv')

### Leave one ligand chain
Since the goal of this script is to find pairs of proteins that bind the same ligands, there is no need to treat 

In [ ]:
# df = df[df["Ligand_chain"]=='A']
# df.shape

value_counts = df["ligand_id"].value_counts()
values_to_keep = value_counts[value_counts > 1].index
log_excluded(df[~df["ligand_id"].isin(values_to_keep)], "singleton_ligand_id", "01_singleton_ligand_excluded.csv")
df = df[df["ligand_id"].isin(values_to_keep)]

filtered_df = df.groupby(["ligand_id", "PDB ID"]).apply(lambda x: x.loc[x['Receptor chain'].idxmin()])
filtered_df = df.groupby(["ligand_id", "PDB ID", "Receptor chain"]).apply(lambda x: x.loc[x['Ligand_chain'].idxmin()])

# remove dna, rna, peptide and none
print("Before filtering:", filtered_df.shape)
non_small_molecule_mask = filtered_df['ligand_id'].str.contains('DNA|RNA|PEPTIDE|NONE', case=False, na=False)
log_excluded(filtered_df[non_small_molecule_mask], "non_small_molecule_ligand_type", "02_ligand_type_excluded.csv")
filtered_df = filtered_df[~non_small_molecule_mask]
print("After filtering:", filtered_df.shape)

filtered_df.shape


In [ ]:
value_counts = df["ligand_id"].value_counts()
values_to_keep = value_counts[value_counts > 1].index
df = df[df["ligand_id"].isin(values_to_keep)]

filtered_df = df.groupby(["ligand_id", "PDB ID"]).apply(lambda x: x.loc[x['Receptor chain'].idxmin()])
filtered_df = df.groupby(["ligand_id", "PDB ID", "Receptor chain"]).apply(lambda x: x.loc[x['Ligand_chain'].idxmin()])

# remove dna, rna, peptide and none
print("Before filtering:", filtered_df.shape)
filtered_df = filtered_df[~filtered_df['ligand_id'].str.contains('DNA|RNA|PEPTIDE|NONE', case=False, na=False)]
print("After filtering:", filtered_df.shape)

filtered_df.shape


In [ ]:
value_counts = df["ligand_id"].value_counts()
pairs_per_ligand = value_counts.apply(lambda x: x*(x-1)//2)
    
# Compute the total number of pairs
total_pairs = pairs_per_ligand.sum()
print("Total number of pairs:", total_pairs)

In [ ]:
def compute_heavy_atoms(smiles):
        smiles_list = smiles.split(';')
        heavy_atoms_counts = []
        for smi in smiles_list:
            if smi[-1] == ';':
                 smi = smi[:-1]
            mol = Chem.MolFromSmiles(smi.strip())  # Strip whitespace around each SMILES string
            if mol is not None:
                heavy_atoms_counts.append(mol.GetNumHeavyAtoms())
        return heavy_atoms_counts


### Counting number of atoms per ligand

In [ ]:
filtered_df.columns
log_excluded(filtered_df[filtered_df['n_ligand_atoms'] < 10], "heavy_atom_count_below_10", "03_heavy_atom_count_excluded.csv")
filtered_df = filtered_df[filtered_df['n_ligand_atoms'] >=10]

In [ ]:
filtered_df.columns
filtered_df = filtered_df[filtered_df['n_ligand_atoms'] >=10]

In [ ]:
column_to_plot = "ligand_id"
value_counts = filtered_df[column_to_plot].value_counts()

n_ligands = 20
# Plot the histogram
plt.bar(value_counts.index[:n_ligands], value_counts.values[:n_ligands], color='blue')
plt.xticks(fontsize=10)  
last_bar_height = value_counts.values[:n_ligands][-1]


plt.xticks(rotation=90)

plt.xlabel(column_to_plot)
plt.ylabel('Frequency')
plt.title(f'Number of pdb chains per ligand (top {n_ligands})')

# Show plot
plt.show()

In [ ]:
# print statistics
print(f"Total number of ligands: {len(filtered_df['ligand_id'].unique())}")
print(f"Total number of PDB IDs: {len(filtered_df['PDB ID'].unique())}")
print(f"Total number of receptor chains: {len(filtered_df['Receptor chain'].unique())}")
print(f"Total number of ligands with heavy atoms >= 10: {len(filtered_df[filtered_df['n_ligand_atoms'] >= 10]['ligand_id'].unique())}")
print(f"Total number of ligands with heavy atoms < 10: {len(filtered_df[filtered_df['n_ligand_atoms'] < 10]['ligand_id'].unique())}")
print(f"Total number of ligands with heavy atoms < 10:          {len(filtered_df[filtered_df['n_ligand_atoms'] < 10])}")

In [ ]:

values = list(ligand_to_n_atoms.values())
plt.hist(values, bins=max(values), edgecolor='black')
plt.axvline(x=10, color='red', linestyle='--', linewidth=1, label='Value = 10')

plt.title('Number of heavy atoms distribution')
plt.xlabel('n atoms')
plt.ylabel('Frequency (number of ligands)')
plt.show()

## Calculate CATH degree

In [ ]:
# cath_chains = pd.read_csv('/home/iscb/wolfson/hagairavid/databases/cath-domain-list.txt',sep='\s+', names=['pdb_chain', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k'], skiprows=16)
cath_chains = pd.read_csv('/home/iscb/wolfson/hagairavid/LocAlign/cath-classification-data/cath-domain-list.txt',sep='\s+', names=['pdb_chain', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k'], skiprows=16)
# cath_chains = cath_chains.set_index('pdb_chain')
cath_chains['domain'] = cath_chains['pdb_chain'].str[-2:]
cath_chains['chain'] = cath_chains['pdb_chain'].str[:-2]

In [ ]:
def compare_rows(row1, row2):
    unequal_indices = np.where(row1 != row2)[0]
    if len(unequal_indices) == 0:
        return len(row1)
    else:
        return unequal_indices[0]

In [ ]:
def compute_cath_degree(combinations, unique_pairs, result_rows, missing_chains):
    
    for row1, row2 in combinations:
        row1 = row1[1]
        row2 = row2[1]

        if row1['PDB ID'] != row2['PDB ID'] and (row1.iloc[0], row2.iloc[0]) not in unique_pairs and (row2.iloc[0], row1.iloc[0]) not in unique_pairs:
            unique_pairs.add((row1.iloc[0], row2.iloc[0]))
            chain1 = f"{row1['PDB ID']}{row1['Receptor chain']}"
            chain2 = f"{row2['PDB ID']}{row2['Receptor chain']}"

            if chain1 in missing_chains or chain2 in missing_chains:
                continue

            chain1_domains = cath_chains[cath_chains['chain'] == chain1]
            if chain1_domains.empty:
                missing_chains.add(chain1)
                continue

            chain2_domains = cath_chains[cath_chains['chain'] == chain2]
            if chain2_domains.empty:
                missing_chains.add(chain2)
                continue

            max_similarity = -1
            for _, domain1 in chain1_domains.iterrows():
                for _, domain2 in chain2_domains.iterrows():
                    max_similarity = max(max_similarity, compare_rows(domain1.values[1:9], domain2.values[1:9]))

            result_rows.append([row1['PDB ID'], row2['PDB ID'], row1['Receptor chain'], row2['Receptor chain'], row2['ligand_id'], max_similarity, heavy_atom_count])
            if len(result_rows) % 10000 == 0:
                print(len(result_rows))

In [ ]:
# take only pairs with CATH degree < 5
log_excluded(result_df[result_df['cath_degree'] >= 5], "cath_degree_ge_5", "04_cath_degree_excluded.csv")
result_df = result_df[result_df['cath_degree'] < 5]

In [ ]:
# take only pairs with CATH degree < 5
result_df = result_df[result_df['cath_degree'] < 5]

In [ ]:
# plot histogram of number of pairs per CATH degree
plt.hist(result_df['cath_degree'], bins=range(6), align='left', edgecolor='black')
plt.xlabel('CATH degree')
plt.ylabel('Frequency')
plt.title('Number of pairs per CATH degree')
plt.xticks(range(5))
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# print statistics.

# n pairs
print(f"Total number of ligands: {len(result_df)}")

# n pairs per CATH degree
print(f"Total number of ligands with CATH degree 0: {len(result_df[result_df['cath_degree'] == 0])}")
print(f"Total number of ligands with CATH degree 1: {len(result_df[result_df['cath_degree'] == 1])}")
print(f"Total number of ligands with CATH degree 2: {len(result_df[result_df['cath_degree'] == 2])}")
print(f"Total number of ligands with CATH degree 3: {len(result_df[result_df['cath_degree'] == 3])}")
print(f"Total number of ligands with CATH degree 4: {len(result_df[result_df['cath_degree'] == 4])}")
# print(f"


In [ ]:
column_to_plot = "ligand_id"
value_counts = result_df[column_to_plot].value_counts()
plt.bar(value_counts.index[:100], value_counts.values[:100], color='blue')
plt.xticks(fontsize=6)  
last_bar_height = value_counts.values[:100][-1]
plt.axhline(y=last_bar_height, color='red', linestyle='--')
plt.text(-10, last_bar_height, f' y={last_bar_height}', color='red', fontsize=10)
plt.xticks(rotation=90)
plt.xlabel(column_to_plot)
plt.ylabel('Frequency')
plt.title(f'{column_to_plot} before sampling')
plt.show()

In [ ]:
ligand_freq = result_df['ligand_id'].value_counts()
cath_degree_freq = result_df['cath_degree'].value_counts()

def calculate_probability(row):
    ligand_prob = 1 / np.sqrt(ligand_freq[row['ligand_id']])
    cath_prob = 1 / np.sqrt(cath_degree_freq[row['cath_degree']])
    return ligand_prob * cath_prob

result_df['Probability'] = result_df.apply(calculate_probability, axis=1)
result_df['Probability'] /= result_df['Probability'].sum()

print(result_df['Probability'])

In [ ]:
sampled_rows = np.random.choice(result_df.index, size=result_df.shape[0], replace=False, p=result_df['Probability'])
sampled_df = result_df.loc[sampled_rows]

In [ ]:
column_to_plot = "ligand_id"
value_counts = sampled_df[column_to_plot].value_counts()

# Plot the histogram
plt.bar(value_counts.index[:20], value_counts.values[:20], color='green' \
'')
plt.xticks(fontsize=10)  # Adjust the font size as per your ptarerence
# Extract the height of the last bar
last_bar_height = value_counts.values[:20][-1]

# plt.axhline(y=last_bar_height, color='red', linestyle='--')
# plt.text(-10, last_bar_height, f' y={last_bar_height}', color='red', fontsize=10)
plt.xticks(rotation=90)
plt.xlabel(column_to_plot)
plt.ylabel('Frequency')
# plt.title(f'{column_to_plot} after sampling')
plt.title(f'Number of pairs per ligand id (top 20 ligands)')

# Show plot
plt.show()

In [ ]:
sampled_df.shape

In [ ]:
import datetime


sampled_df.head()
date = datetime.datetime.now().strftime("%d_%m")
sampled_df.to_csv(os.path.join('calculated_cath',f'sampled_{sampled_df.shape[0]}_{date}.csv'))